# 🌱 KrishiTwin — XGBoost with Walk-Forward CV
## Data Leakage Fix + Yearly Pattern Learning
---
### Strategy:
- **Walk-Forward Split** → Model sirf past data dekhega, future nahi
- **Lag Features** → Model ko explicitly year-over-year pattern sikhate hain
- **Log Transform** → Skewed yield distributions fix
- **No Shuffle** → Time order hamesha preserve hoga

## CELL 1 — Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib
import warnings
warnings.filterwarnings('ignore')

!pip install xgboost
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error

print('✅ Libraries Loaded')

✅ Libraries Loaded


## CELL 2 — File Path (Yahan Apna Path Dalo)

In [3]:
# ============================================================
# 🔧 APNA FILE PATH YAHAN DALO
# ============================================================
DATA_PATH = '/content/KrishiTwin_Final_Engineered.csv'   # << Apna path
# ============================================================

df = pd.read_csv(DATA_PATH)
df.columns = [c.replace('..', '.').strip() for c in df.columns]

# Standardize common column name variants
rename_map = {
    'State.Name' : 'State Name',
    'dist.name'  : 'dist_name',
    'dist.code'  : 'dist_code',
}
df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

df = df.sort_values(['dist_code', 'year']).reset_index(drop=True)

print(f'✅ Data Loaded: {df.shape}')
print(f'   Year Range : {df["year"].min()} – {df["year"].max()}')
print(f'   Districts  : {df["dist_code"].nunique()}')
print(f'   States     : {df["State Name"].nunique()}')

✅ Data Loaded: (7841, 35)
   Year Range : 1990 – 2015
   Districts  : 302
   States     : 20


In [11]:
df

,dist_code,year,State Name,dist_name,RICE YIELD (Kg per ha),PEARL MILLET YIELD (Kg per ha),CHICKPEA YIELD (Kg per ha),GROUNDNUT YIELD (Kg per ha),SUGARCANE YIELD (Kg per ha),NPK_Intensity_KgHa,Irrigation_Intensity_Ratio,WDI,Kharif_Avg_MaxTemp,Kharif_Total_Rain,Rabi_Avg_MaxTemp,District_Soil_Health_Score,WHEAT.YIELD.Kg.per.ha.,KHARIF.SORGHUM.YIELD.Kg.per.ha.,RABI.SORGHUM.YIELD.Kg.per.ha.,SORGHUM.YIELD.Kg.per.ha.,PEARL.MILLET.YIELD.Kg.per.ha.,MAIZE.YIELD.Kg.per.ha.,FINGER.MILLET.YIELD.Kg.per.ha.,BARLEY.YIELD.Kg.per.ha.,PIGEONPEA.YIELD.Kg.per.ha.,MINOR.PULSES.YIELD.Kg.per.ha.,SESAMUM.YIELD.Kg.per.ha.,RAPESEED.AND.MUSTARD.YIELD.Kg.per.ha.,SAFFLOWER.YIELD.Kg.per.ha.,CASTOR.YIELD.Kg.per.ha.,LINSEED.YIELD.Kg.per.ha.,SUNFLOWER.YIELD.Kg.per.ha.,SOYABEAN.YIELD.Kg.per.ha.,OILSEEDS.YIELD.Kg.per.ha.,COTTON.YIELD.Kg.per.ha.
0,1,1990,chhattisgarh,durg,1210.0,0.0,572.0,848.0,0.0,25.758575,0.226781,0.524960,31.112,1490.460007,29.335000,22.533164,685.08,800.00,0.00,800.00,0.0,950.00,0.0,1000.0,1031.58,349.87,157.89,574.55,0.0,0.0,181.68,0.00,778.33,233.33,0.0
1,1,1991,chhattisgarh,durg,1293.0,0.0,690.0,1040.0,1190.0,34.017668,0.270240,0.518565,31.796,1138.270012,28.800000,22.533164,620.69,782.61,0.00,782.61,0.0,921.05,0.0,0.0,1041.10,499.44,142.86,576.35,500.0,0.0,261.43,250.00,657.68,300.93,0.0
2,1,1992,chhattisgarh,durg,1291.0,0.0,626.0,1438.0,1667.0,42.239316,0.281473,0.523945,32.140,1027.639984,28.380000,22.533164,365.42,909.09,0.00,909.09,0.0,1363.64,0.0,0.0,1035.09,388.52,166.67,601.27,250.0,0.0,226.16,222.22,705.88,310.57,0.0
3,1,1993,chhattisgarh,durg,1387.0,0.0,684.0,1042.0,2500.0,32.244130,0.280058,0.515183,32.224,1214.430000,28.677500,22.533164,704.14,1000.00,1416.67,1384.62,1000.0,1105.26,0.0,0.0,1086.71,428.48,166.67,504.79,250.0,0.0,208.43,304.35,887.81,389.69,0.0
4,1,1994,chhattisgarh,durg,1399.0,0.0,725.0,1000.0,1000.0,35.144383,0.284562,0.525718,31.282,1593.490021,28.695000,22.533164,805.48,666.67,0.00,666.67,0.0,895.83,0.0,0.0,1019.23,402.91,137.93,620.69,250.0,0.0,248.92,320.00,792.90,341.88,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7836,916,2011,jharkhand,ranchi,2590.0,0.0,1074.0,0.0,0.0,282.163257,0.112560,0.736846,30.352,1480.900032,26.100000,77.037681,1701.25,0.00,0.00,0.00,0.0,1909.86,968.1,0.0,1540.74,1000.00,0.00,660.00,0.0,0.0,0.00,0.00,0.00,660.00,0.0
7837,916,2012,jharkhand,ranchi,2509.0,0.0,1042.0,0.0,0.0,191.368678,0.139654,0.758243,30.928,1259.120022,25.222500,77.037681,1815.63,0.00,0.00,0.00,0.0,2116.31,0.0,0.0,1820.51,1000.00,0.00,956.83,0.0,0.0,0.00,0.00,0.00,956.83,0.0
7838,916,2013,jharkhand,ranchi,2340.0,0.0,1367.0,0.0,0.0,117.456673,0.123408,0.735018,30.408,1371.509995,24.980000,77.037681,1941.18,0.00,0.00,0.00,0.0,1936.72,0.0,0.0,1693.25,777.78,0.00,861.54,0.0,0.0,0.00,0.00,0.00,861.54,0.0
7839,916,2014,jharkhand,ranchi,2608.0,0.0,1021.0,0.0,0.0,217.122830,0.139654,0.719373,30.862,1021.559975,23.980000,77.037681,1806.02,0.00,0.00,0.00,0.0,2261.41,0.0,0.0,1901.41,1290.08,0.00,912.09,0.0,0.0,0.00,0.00,0.00,912.09,0.0


In [5]:
df['dist_name'] = df['dist_name'].str.lower().str.strip()
df['State Name'] = df['State Name'].str.lower().str.strip()

In [6]:
dist_mapping = dict(zip(df['dist_name'], df['dist_code']))

In [ ]:
dist_mapping

{'durg': 1,
 'bastar': 2,
 'raipur': 3,
 'bilaspur': 4,
 'raigarh': 5,
 'surguja': 6,
 'jabalpur': 7,
 'balaghat': 8,
 'chhindwara': 9,
 'narsinghpur': 10,
 'seoni / shivani': 11,
 'mandla': 12,
 'sagar': 13,
 'damoh': 14,
 'tikamgarh': 15,
 'chhatarpur': 16,
 'panna': 17,
 'rewa': 18,
 'sidhi': 19,
 'satna': 20,
 'shahdol': 21,
 'gwalior': 22,
 'shivpuri': 23,
 'guna': 24,
 'datia': 25,
 'morena': 26,
 'bhind': 27,
 'indore': 28,
 'ratlam': 29,
 'ujjain': 30,
 'mandsaur': 31,
 'dewas': 32,
 'dhar': 33,
 'jhabua': 34,
 'khargone / west nimar': 35,
 'khandwa / east nimar': 36,
 'sehore': 37,
 'raisen': 38,
 'vidisha': 39,
 'betul': 40,
 'rajgarh': 41,
 'shajapur': 42,
 'hoshangabad': 43,
 'srikakulam': 44,
 'visakhapatnam': 45,
 'east godavari': 46,
 'west godavari': 47,
 'krishna': 48,
 'guntur': 49,
 's.p.s. nellore': 50,
 'kurnool': 51,
 'ananthapur': 52,
 'kadapa ysr': 53,
 'chittoor': 54,
 'hyderabad': 55,
 'nizamabad': 56,
 'medak': 57,
 'mahabubnagar': 58,
 'nalgonda': 59,
 'wara

## CELL 3 — Feature Engineering
### Agar already engineered CSV hai to CELL 3 skip karo aur CELL 4 se shuru karo

In [7]:
# ── Sirf tab run karo agar raw (non-engineered) CSV load kiya hai ──────────
# Agar tumhara CSV already KrishiTwin_Final_Engineered.csv hai
# (NPK_Intensity_KgHa, WDI etc. already present hain) toh ye cell skip karo

RUN_FE = False   # << True karo agar raw CSV load kiya hai

if RUN_FE:
    mendeley_crops = ['RICE', 'PEARL MILLET', 'CHICKPEA', 'GROUNDNUT', 'SUGARCANE']

    # Area/Crop cols may have dot-notation
    def get_col(df, keyword):
        matches = [c for c in df.columns if keyword.upper() in c.upper()]
        return matches[0] if matches else None

    total_fert  = get_col(df, 'TOTAL FERTILISER')
    gross_crop  = get_col(df, 'GROSS CROPPED AREA')
    gross_irr   = get_col(df, 'GROSS IRRIGATED AREA')
    rice_area   = get_col(df, 'RICE AREA')
    sugar_area  = get_col(df, 'SUGARCANE AREA')
    rice_yield  = get_col(df, 'RICE YIELD')

    df['NPK_Intensity_KgHa']       = df[total_fert] / df[gross_crop]
    df['Irrigation_Intensity_Ratio']= df[gross_irr]  / df[gross_crop]
    df['WDI']                       = (df[rice_area] + df[sugar_area]) / df[gross_crop]

    kharif_temp_cols = [c for c in df.columns if any(m in c for m in ['JUNE','JULY','AUGUST','SEPTEMBER','OCTOBER']) and 'MAXIMUM' in c and 'Winter' not in c and 'Rainy' not in c]
    kharif_rain_cols = [c for c in df.columns if any(m in c for m in ['JUNE','JULY','AUGUST','SEPTEMBER','OCTOBER']) and 'PERCIPITATION' in c and 'Rainy' not in c]
    rabi_temp_cols   = [c for c in df.columns if any(m in c for m in ['NOVEMBER','DECEMBER','JANUARY','FEBRUARY']) and 'MAXIMUM' in c and 'Winter' not in c]

    df['Kharif_Avg_MaxTemp'] = df[kharif_temp_cols].mean(axis=1)
    df['Kharif_Total_Rain']  = df[kharif_rain_cols].sum(axis=1)
    df['Rabi_Avg_MaxTemp']   = df[rabi_temp_cols].mean(axis=1)

    df['Soil_Absorption_Efficiency'] = df[rice_yield] / (df['NPK_Intensity_KgHa'] + 1)
    df['District_Soil_Health_Score'] = df.groupby('dist_code')['Soil_Absorption_Efficiency'].transform('mean')

    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    print('✅ Feature Engineering Done')
else:
    print('⏭️  FE Skipped — using pre-engineered CSV')

⏭️  FE Skipped — using pre-engineered CSV


In [10]:
df
pd.set_option('display.max_columns', None)

## CELL 4 — Lag Features
### Yearly Pattern sikhane ka core — Model ab year-over-year trend dekhega
```
Kyun Lag Features?
────────────────────────────────────────────────────────
XGBoost time ki sequence nahi jaanta.
Lag features explicitly batate hain:
  'Pichle saal NPK itna tha, usse pehle itna'
  'Temperature trend kya tha'
  'Rainfall 3 saal se badh rahi ya ghat rahi'
Tab model sikhta hai ki yearly changes yield ko kaise affect karti hain.
────────────────────────────────────────────────────────
```

In [12]:
# ── Columns jinke lag features banana hai ─────────────────────────────────
LAG_COLS = [
    'NPK_Intensity_KgHa',
    'Irrigation_Intensity_Ratio',
    'WDI',
    'Kharif_Avg_MaxTemp',
    'Kharif_Total_Rain',
    'Rabi_Avg_MaxTemp',
]

# ── Kitne saal ka lag chahiye ──────────────────────────────────────────────
LAG_YEARS = [1, 2, 3]   # Year-1, Year-2, Year-3

df = df.sort_values(['dist_code', 'year']).reset_index(drop=True)

for col in LAG_COLS:
    if col not in df.columns:
        print(f'⚠️  Column not found, skipping lag: {col}')
        continue
    for lag in LAG_YEARS:
        lag_col_name = f'{col}_Lag{lag}'
        # groupby dist_code so that lag doesn't bleed across districts
        df[lag_col_name] = df.groupby('dist_code')[col].shift(lag)

# ── YoY Change Features — trend direction batate hain ─────────────────────
# Eg: Kharif_Total_Rain_Delta1 = Rain(t) - Rain(t-1)
DELTA_COLS = ['Kharif_Avg_MaxTemp', 'Kharif_Total_Rain', 'NPK_Intensity_KgHa']
for col in DELTA_COLS:
    if col in df.columns:
        lag1 = f'{col}_Lag1'
        if lag1 in df.columns:
            df[f'{col}_Delta1'] = df[col] - df[lag1]

# ── 3-Year Rolling Mean — smooth trend ────────────────────────────────────
ROLLING_COLS = ['Kharif_Avg_MaxTemp', 'Kharif_Total_Rain']
for col in ROLLING_COLS:
    if col in df.columns:
        df[f'{col}_Roll3'] = (
            df.groupby('dist_code')[col]
              .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
        )
        # Note: shift(1) pehle kiya — current year ka data leak na ho

# ── Drop rows jahan lag values NaN hain (first 3 years per district) ──────
lag_feature_cols = [f'{c}_Lag{l}' for c in LAG_COLS for l in LAG_YEARS if c in df.columns]
df_lagged = df.dropna(subset=lag_feature_cols).reset_index(drop=True)

print(f'✅ Lag Features Created')
print(f'   Rows before lag drop : {len(df):,}')
print(f'   Rows after lag drop  : {len(df_lagged):,}  (first 3 yrs per district removed — expected)')
print(f'   New lag columns      : {len(lag_feature_cols)}')

✅ Lag Features Created
   Rows before lag drop : 7,841
   Rows after lag drop  : 6,935  (first 3 yrs per district removed — expected)
   New lag columns      : 18


In [13]:
df_lagged.shape

(6935, 58)

In [14]:
df_lagged

,dist_code,year,State Name,dist_name,RICE YIELD (Kg per ha),PEARL MILLET YIELD (Kg per ha),CHICKPEA YIELD (Kg per ha),GROUNDNUT YIELD (Kg per ha),SUGARCANE YIELD (Kg per ha),NPK_Intensity_KgHa,Irrigation_Intensity_Ratio,WDI,Kharif_Avg_MaxTemp,Kharif_Total_Rain,Rabi_Avg_MaxTemp,District_Soil_Health_Score,WHEAT.YIELD.Kg.per.ha.,KHARIF.SORGHUM.YIELD.Kg.per.ha.,RABI.SORGHUM.YIELD.Kg.per.ha.,SORGHUM.YIELD.Kg.per.ha.,PEARL.MILLET.YIELD.Kg.per.ha.,MAIZE.YIELD.Kg.per.ha.,FINGER.MILLET.YIELD.Kg.per.ha.,BARLEY.YIELD.Kg.per.ha.,PIGEONPEA.YIELD.Kg.per.ha.,MINOR.PULSES.YIELD.Kg.per.ha.,SESAMUM.YIELD.Kg.per.ha.,RAPESEED.AND.MUSTARD.YIELD.Kg.per.ha.,SAFFLOWER.YIELD.Kg.per.ha.,CASTOR.YIELD.Kg.per.ha.,LINSEED.YIELD.Kg.per.ha.,SUNFLOWER.YIELD.Kg.per.ha.,SOYABEAN.YIELD.Kg.per.ha.,OILSEEDS.YIELD.Kg.per.ha.,COTTON.YIELD.Kg.per.ha.,NPK_Intensity_KgHa_Lag1,NPK_Intensity_KgHa_Lag2,NPK_Intensity_KgHa_Lag3,Irrigation_Intensity_Ratio_Lag1,Irrigation_Intensity_Ratio_Lag2,Irrigation_Intensity_Ratio_Lag3,WDI_Lag1,WDI_Lag2,WDI_Lag3,Kharif_Avg_MaxTemp_Lag1,Kharif_Avg_MaxTemp_Lag2,Kharif_Avg_MaxTemp_Lag3,Kharif_Total_Rain_Lag1,Kharif_Total_Rain_Lag2,Kharif_Total_Rain_Lag3,Rabi_Avg_MaxTemp_Lag1,Rabi_Avg_MaxTemp_Lag2,Rabi_Avg_MaxTemp_Lag3,Kharif_Avg_MaxTemp_Delta1,Kharif_Total_Rain_Delta1,NPK_Intensity_KgHa_Delta1,Kharif_Avg_MaxTemp_Roll3,Kharif_Total_Rain_Roll3
0,1,1993,chhattisgarh,durg,1387.0,0.0,684.0,1042.0,2500.0,32.244130,0.280058,0.515183,32.224,1214.430000,28.677500,22.533164,704.14,1000.00,1416.67,1384.62,1000.0,1105.26,0.0,0.0,1086.71,428.48,166.67,504.79,250.00,0.0,208.43,304.35,887.81,389.69,0.0,42.239316,34.017668,25.758575,0.281473,0.270240,0.226781,0.523945,0.518565,0.524960,32.140,31.796,31.112,1027.639984,1138.270012,1490.460007,28.380000,28.800000,29.335000,0.084000,186.790016,-9.995187,31.682667,1218.790001
1,1,1994,chhattisgarh,durg,1399.0,0.0,725.0,1000.0,1000.0,35.144383,0.284562,0.525718,31.282,1593.490021,28.695000,22.533164,805.48,666.67,0.00,666.67,0.0,895.83,0.0,0.0,1019.23,402.91,137.93,620.69,250.00,0.0,248.92,320.00,792.90,341.88,0.0,32.244130,42.239316,34.017668,0.280058,0.281473,0.270240,0.515183,0.523945,0.518565,32.224,32.140,31.796,1214.430000,1027.639984,1138.270012,28.677500,28.380000,28.800000,-0.942000,379.060020,2.900253,32.053333,1126.779999
2,1,1995,chhattisgarh,durg,1507.0,0.0,497.0,1125.0,1000.0,35.180497,0.287350,0.518796,32.528,1192.210018,28.632500,22.533164,714.29,1000.00,0.00,1000.00,0.0,1333.33,0.0,1000.0,854.30,253.92,186.05,481.48,666.67,0.0,172.10,333.33,864.75,328.14,0.0,35.144383,32.244130,42.239316,0.284562,0.280058,0.281473,0.525718,0.515183,0.523945,31.282,32.224,32.140,1593.490021,1214.430000,1027.639984,28.695000,28.677500,28.380000,1.246000,-401.280003,0.036114,31.882000,1278.520002
3,1,1996,chhattisgarh,durg,1486.0,0.0,760.0,1000.0,2000.0,35.769900,0.286679,0.506468,32.142,901.209999,29.342500,22.533164,803.87,666.67,0.00,666.67,1000.0,1281.25,0.0,0.0,976.19,443.26,155.56,500.00,500.00,0.0,273.94,333.33,688.89,422.60,0.0,35.180497,35.144383,32.244130,0.287350,0.284562,0.280058,0.518796,0.525718,0.515183,32.528,31.282,32.224,1192.210018,1593.490021,1214.430000,28.632500,28.695000,28.677500,-0.386000,-291.000019,0.589403,32.011333,1333.376680
4,1,1997,chhattisgarh,durg,1265.0,0.0,364.0,1091.0,1613.0,48.123741,0.293813,0.545439,32.280,888.940025,28.820000,22.533164,514.04,82.64,0.00,80.65,0.0,914.29,0.0,0.0,595.04,40.02,133.33,384.62,333.33,0.0,130.92,250.00,793.83,417.56,0.0,35.769900,35.180497,35.144383,0.286679,0.287350,0.284562,0.506468,0.518796,0.525718,32.142,32.528,31.282,901.209999,1192.210018,1593.490021,29.342500,28.632500,28.695000,0.138000,-12.269974,12.353841,31.984000,1228.970013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6930,916,2011,jharkhand,ranchi,2590.0,0.0,1074.0,0.0,0.0,282.163257,0.112560,0

## CELL 5 — Melt to Long Format + Encode

In [15]:
# ── Yield columns identify karo ───────────────────────────────────────────
all_yield_cols = [c for c in df_lagged.columns if 'YIELD' in c.upper()]

# Pearl Millet duplicate remove
yield_cols = []
for c in all_yield_cols:
    if 'PEARL' in c.upper() and 'MILLET' in c.upper():
        if c == 'PEARL MILLET YIELD (Kg per ha)':
            yield_cols.append(c)
    else:
        yield_cols.append(c)

print(f'✅ Yield columns selected: {len(yield_cols)}')

# ── ID cols — sab features jo melt ke baad bhi chahiye ────────────────────
base_features = [
    'NPK_Intensity_KgHa', 'Irrigation_Intensity_Ratio', 'WDI',
    'Kharif_Avg_MaxTemp', 'Kharif_Total_Rain', 'Rabi_Avg_MaxTemp',
    'District_Soil_Health_Score'
]
lag_features  = [c for c in df_lagged.columns if '_Lag' in c or '_Delta' in c or '_Roll' in c]

id_cols = ['dist_code', 'year', 'State Name'] + base_features + lag_features
id_cols = [c for c in id_cols if c in df_lagged.columns]  # safety check

# ── Melt ──────────────────────────────────────────────────────────────────
df_melted = df_lagged.melt(
    id_vars=id_cols,
    value_vars=yield_cols,
    var_name='Crop_Type',
    value_name='Yield'
)
print(f'✅ Data Melted: {df_melted.shape}')
df_melted = df_melted[df_melted['Yield'] > 0].reset_index(drop=True)
print(f'✅ After filtering non-positive yields: {df_melted.shape}')

# ── Label Encoding ────────────────────────────────────────────────────────
le_crop  = LabelEncoder()
le_state = LabelEncoder()
df_melted['Crop_Encoded']  = le_crop.fit_transform(df_melted['Crop_Type'])
df_melted['State_Encoded'] = le_state.fit_transform(df_melted['State Name'])

# Save encoders
joblib.dump(le_crop,  'crop_encoder.pkl')
joblib.dump(le_state, 'state_encoder.pkl')

print(f'✅ Melted shape     : {df_melted.shape}')
print(f'   Unique crops    : {df_melted["Crop_Type"].nunique()}')
print(f'   Year range      : {df_melted["year"].min()} – {df_melted["year"].max()}')
print(f'   Mean Yield      : {df_melted["Yield"].mean():.1f} Kg/ha')
print('✅ Encoders saved: crop_encoder.pkl, state_encoder.pkl')

✅ Yield columns selected: 23
✅ Data Melted: (159505, 35)
✅ After filtering non-positive yields: (101907, 35)
✅ Melted shape     : (101907, 37)
   Unique crops    : 23
   Year range      : 1993 – 2015
   Mean Yield      : 1325.4 Kg/ha
✅ Encoders saved: crop_encoder.pkl, state_encoder.pkl


In [16]:
df_melted.head()

,dist_code,year,State Name,NPK_Intensity_KgHa,Irrigation_Intensity_Ratio,WDI,Kharif_Avg_MaxTemp,Kharif_Total_Rain,Rabi_Avg_MaxTemp,District_Soil_Health_Score,NPK_Intensity_KgHa_Lag1,NPK_Intensity_KgHa_Lag2,NPK_Intensity_KgHa_Lag3,Irrigation_Intensity_Ratio_Lag1,Irrigation_Intensity_Ratio_Lag2,Irrigation_Intensity_Ratio_Lag3,WDI_Lag1,WDI_Lag2,WDI_Lag3,Kharif_Avg_MaxTemp_Lag1,Kharif_Avg_MaxTemp_Lag2,Kharif_Avg_MaxTemp_Lag3,Kharif_Total_Rain_Lag1,Kharif_Total_Rain_Lag2,Kharif_Total_Rain_Lag3,Rabi_Avg_MaxTemp_Lag1,Rabi_Avg_MaxTemp_Lag2,Rabi_Avg_MaxTemp_Lag3,Kharif_Avg_MaxTemp_Delta1,Kharif_Total_Rain_Delta1,NPK_Intensity_KgHa_Delta1,Kharif_Avg_MaxTemp_Roll3,Kharif_Total_Rain_Roll3,Crop_Type,Yield,Crop_Encoded,State_Encoded
0,1,1993,chhattisgarh,32.244130,0.280058,0.515183,32.224,1214.430000,28.6775,22.533164,42.239316,34.017668,25.758575,0.281473,0.270240,0.226781,0.523945,0.518565,0.524960,32.140,31.796,31.112,1027.639984,1138.270012,1490.460007,28.3800,28.8000,29.3350,0.084,186.790016,-9.995187,31.682667,1218.790001,RICE YIELD (Kg per ha),1387.0,15,3
1,1,1994,chhattisgarh,35.144383,0.284562,0.525718,31.282,1593.490021,28.6950,22.533164,32.244130,42.239316,34.017668,0.280058,0.281473,0.270240,0.515183,0.523945,0.518565,32.224,32.140,31.796,1214.430000,1027.639984,1138.270012,28.6775,28.3800,28.8000,-0.942,379.060020,2.900253,32.053333,1126.779999,RICE YIELD (Kg per ha),1399.0,15,3
2,1,1995,chhattisgarh,35.180497,0.287350,0.518796,32.528,1192.210018,28.6325,22.533164,35.144383,32.244130,42.239316,0.284562,0.280058,0.281473,0.525718,0.515183,0.523945,31.282,32.224,32.140,1593.490021,1214.430000,1027.639984,28.6950,28.6775,28.3800,1.246,-401.280003,0.036114,31.882000,1278.520002,RICE YIELD (Kg per ha),1507.0,15,3
3,1,1996,chhattisgarh,35.769900,0.286679,0.506468,32.142,901.209999,29.3425,22.533164,35.180497,35.144383,32.244130,0.287350,0.284562,0.280058,0.518796,0.525718,0.515183,32.528,31.282,32.224,1192.210018,1593.490021,1214.430000,28.6325,28.6950,28.6775,-0.386,-291.000019,0.589403,32.011333,1333.376680,RICE YIELD (Kg per ha),1486.0,15,3
4,1,1997,chhattisgarh,48.123741,0.293813,0.545439,32.280,888.940025,28.8200,22.533164,35.769900,35.180497,35.144383,0.286679,0.287350,0.284562,0.506468,0.518796,0.525718,32.142,32.528,31.282,901.209999,1192.210018,1593.490021,29.3425,28.6325,28.6950,0.138,-12.269974,12.353841,31.984000,1228.970013,RICE YIELD (Kg per ha),1265.0,15,3


In [17]:
for i, crop in enumerate(le_crop.classes_):
    print(f"{crop} → {i}")

BARLEY.YIELD.Kg.per.ha. → 0
CASTOR.YIELD.Kg.per.ha. → 1
CHICKPEA YIELD (Kg per ha) → 2
COTTON.YIELD.Kg.per.ha. → 3
FINGER.MILLET.YIELD.Kg.per.ha. → 4
GROUNDNUT YIELD (Kg per ha) → 5
KHARIF.SORGHUM.YIELD.Kg.per.ha. → 6
LINSEED.YIELD.Kg.per.ha. → 7
MAIZE.YIELD.Kg.per.ha. → 8
MINOR.PULSES.YIELD.Kg.per.ha. → 9
OILSEEDS.YIELD.Kg.per.ha. → 10
PEARL MILLET YIELD (Kg per ha) → 11
PIGEONPEA.YIELD.Kg.per.ha. → 12
RABI.SORGHUM.YIELD.Kg.per.ha. → 13
RAPESEED.AND.MUSTARD.YIELD.Kg.per.ha. → 14
RICE YIELD (Kg per ha) → 15
SAFFLOWER.YIELD.Kg.per.ha. → 16
SESAMUM.YIELD.Kg.per.ha. → 17
SORGHUM.YIELD.Kg.per.ha. → 18
SOYABEAN.YIELD.Kg.per.ha. → 19
SUGARCANE YIELD (Kg per ha) → 20
SUNFLOWER.YIELD.Kg.per.ha. → 21
WHEAT.YIELD.Kg.per.ha. → 22


In [18]:
for i, state in enumerate(le_state.classes_):
    print(f"{state} → {i}")

andhra pradesh → 0
assam → 1
bihar → 2
chhattisgarh → 3
gujarat → 4
haryana → 5
himachal pradesh → 6
jharkhand → 7
karnataka → 8
kerala → 9
madhya pradesh → 10
maharashtra → 11
orissa → 12
punjab → 13
rajasthan → 14
tamil nadu → 15
telangana → 16
uttar pradesh → 17
uttarakhand → 18
west bengal → 19


## CELL 6 — Walk-Forward Cross Validation
```
Walk-Forward Split kaise kaam karta hai:
────────────────────────────────────────────────────────────────
Fold 1:  Train = 1993–2005  │  Test = 2006
Fold 2:  Train = 1993–2006  │  Test = 2007
Fold 3:  Train = 1993–2007  │  Test = 2008
  ...         ...           │    ...
Fold N:  Train = 1993–2014  │  Test = 2015

✅ Model kabhi future data nahi dekhta
✅ Har fold mein window badh ti jaati hai
✅ Real production scenario ka perfect simulation
────────────────────────────────────────────────────────────────
```

In [19]:
def walk_forward_split(df, year_col='year', min_train_years=10, test_window=1):
    """
    Walk-Forward splits generate karta hai.

    min_train_years : minimum kitne saal ka train data chahiye
    test_window     : har fold mein kitne saal ka test (1 = single year)

    Returns list of (train_idx, test_idx) tuples.
    """
    years     = sorted(df[year_col].unique())
    min_year  = years[0]
    splits    = []

    for i in range(min_train_years, len(years), test_window):
        test_years  = years[i : i + test_window]
        train_years = years[:i]

        if not test_years:
            break

        train_idx = df[df[year_col].isin(train_years)].index
        test_idx  = df[df[year_col].isin(test_years)].index

        splits.append((train_idx, test_idx))

    return splits, years


splits, all_years = walk_forward_split(df_melted, year_col='year', min_train_years=10)

print(f'✅ Walk-Forward Splits Generated: {len(splits)} folds')
print(f'   Training starts from year    : {all_years[0]}')
print()
print(f'   {"Fold":<6} {"Train Range":<20} {"Test Year"}')
print('   ' + '-' * 40)
for i, (tr, te) in enumerate(splits):
    train_yrs = df_melted.loc[tr, 'year']
    test_yrs  = df_melted.loc[te, 'year']
    print(f'   {i+1:<6} {train_yrs.min()}–{train_yrs.max():<15} {test_yrs.min()}–{test_yrs.max()}')

✅ Walk-Forward Splits Generated: 13 folds
   Training starts from year    : 1993

   Fold   Train Range          Test Year
   ----------------------------------------
   1      1993–2002            2003–2003
   2      1993–2003            2004–2004
   3      1993–2004            2005–2005
   4      1993–2005            2006–2006
   5      1993–2006            2007–2007
   6      1993–2007            2008–2008
   7      1993–2008            2009–2009
   8      1993–2009            2010–2010
   9      1993–2010            2011–2011
   10     1993–2011            2012–2012
   11     1993–2012            2013–2013
   12     1993–2013            2014–2014
   13     1993–2014            2015–2015


## CELL 7 — Feature List (Lag + Base Features)

In [20]:
# ── Base features ─────────────────────────────────────────────────────────
base_model_features = [
    'year',
    'State_Encoded',
    'Crop_Encoded',
    'NPK_Intensity_KgHa',
    'Irrigation_Intensity_Ratio',
    'WDI',
    'Kharif_Avg_MaxTemp',
    'Kharif_Total_Rain',
    'Rabi_Avg_MaxTemp',
    'District_Soil_Health_Score',
]

# ── Lag + Delta + Rolling features ────────────────────────────────────────
# Ye explicitly yearly pattern sikhate hain XGBoost ko
lag_model_features = [c for c in df_melted.columns
                      if ('_Lag' in c or '_Delta' in c or '_Roll' in c)
                      and c in df_melted.columns]

ALL_FEATURES = base_model_features + lag_model_features
# Safety: sirf wo features jo actually df_melted mein hain
ALL_FEATURES = [f for f in ALL_FEATURES if f in df_melted.columns]

print(f'✅ Total features for model: {len(ALL_FEATURES)}')
print(f'   Base features : {len(base_model_features)}')
print(f'   Lag+Delta+Roll: {len(lag_model_features)}')
print()
print('Feature List:')
for f in ALL_FEATURES:
    print(f'   {f}')

✅ Total features for model: 33
   Base features : 10
   Lag+Delta+Roll: 23

Feature List:
   year
   State_Encoded
   Crop_Encoded
   NPK_Intensity_KgHa
   Irrigation_Intensity_Ratio
   WDI
   Kharif_Avg_MaxTemp
   Kharif_Total_Rain
   Rabi_Avg_MaxTemp
   District_Soil_Health_Score
   NPK_Intensity_KgHa_Lag1
   NPK_Intensity_KgHa_Lag2
   NPK_Intensity_KgHa_Lag3
   Irrigation_Intensity_Ratio_Lag1
   Irrigation_Intensity_Ratio_Lag2
   Irrigation_Intensity_Ratio_Lag3
   WDI_Lag1
   WDI_Lag2
   WDI_Lag3
   Kharif_Avg_MaxTemp_Lag1
   Kharif_Avg_MaxTemp_Lag2
   Kharif_Avg_MaxTemp_Lag3
   Kharif_Total_Rain_Lag1
   Kharif_Total_Rain_Lag2
   Kharif_Total_Rain_Lag3
   Rabi_Avg_MaxTemp_Lag1
   Rabi_Avg_MaxTemp_Lag2
   Rabi_Avg_MaxTemp_Lag3
   Kharif_Avg_MaxTemp_Delta1
   Kharif_Total_Rain_Delta1
   NPK_Intensity_KgHa_Delta1
   Kharif_Avg_MaxTemp_Roll3
   Kharif_Total_Rain_Roll3


In [21]:
df_melted.shape

(101907, 37)

In [22]:
df_melted.head()
pd.set_option('display.max_columns', None)

In [ ]:
df_melted.head()

,dist_code,year,State Name,NPK_Intensity_KgHa,Irrigation_Intensity_Ratio,WDI,Kharif_Avg_MaxTemp,Kharif_Total_Rain,Rabi_Avg_MaxTemp,District_Soil_Health_Score,NPK_Intensity_KgHa_Lag1,NPK_Intensity_KgHa_Lag2,NPK_Intensity_KgHa_Lag3,Irrigation_Intensity_Ratio_Lag1,Irrigation_Intensity_Ratio_Lag2,Irrigation_Intensity_Ratio_Lag3,WDI_Lag1,WDI_Lag2,WDI_Lag3,Kharif_Avg_MaxTemp_Lag1,Kharif_Avg_MaxTemp_Lag2,Kharif_Avg_MaxTemp_Lag3,Kharif_Total_Rain_Lag1,Kharif_Total_Rain_Lag2,Kharif_Total_Rain_Lag3,Rabi_Avg_MaxTemp_Lag1,Rabi_Avg_MaxTemp_Lag2,Rabi_Avg_MaxTemp_Lag3,Kharif_Avg_MaxTemp_Delta1,Kharif_Total_Rain_Delta1,NPK_Intensity_KgHa_Delta1,Kharif_Avg_MaxTemp_Roll3,Kharif_Total_Rain_Roll3,Crop_Type,Yield,Crop_Encoded,State_Encoded
0,1,1993,chhattisgarh,32.244130,0.280058,0.515183,32.224,1214.430000,28.6775,22.533164,42.239316,34.017668,25.758575,0.281473,0.270240,0.226781,0.523945,0.518565,0.524960,32.140,31.796,31.112,1027.639984,1138.270012,1490.460007,28.3800,28.8000,29.3350,0.084,186.790016,-9.995187,31.682667,1218.790001,RICE YIELD (Kg per ha),1387.0,15,3
1,1,1994,chhattisgarh,35.144383,0.284562,0.525718,31.282,1593.490021,28.6950,22.533164,32.244130,42.239316,34.017668,0.280058,0.281473,0.270240,0.515183,0.523945,0.518565,32.224,32.140,31.796,1214.430000,1027.639984,1138.270012,28.6775,28.3800,28.8000,-0.942,379.060020,2.900253,32.053333,1126.779999,RICE YIELD (Kg per ha),1399.0,15,3
2,1,1995,chhattisgarh,35.180497,0.287350,0.518796,32.528,1192.210018,28.6325,22.533164,35.144383,32.244130,42.239316,0.284562,0.280058,0.281473,0.525718,0.515183,0.523945,31.282,32.224,32.140,1593.490021,1214.430000,1027.639984,28.6950,28.6775,28.3800,1.246,-401.280003,0.036114,31.882000,1278.520002,RICE YIELD (Kg per ha),1507.0,15,3
3,1,1996,chhattisgarh,35.769900,0.286679,0.506468,32.142,901.209999,29.3425,22.533164,35.180497,35.144383,32.244130,0.287350,0.284562,0.280058,0.518796,0.525718,0.515183,32.528,31.282,32.224,1192.210018,1593.490021,1214.430000,28.6325,28.6950,28.6775,-0.386,-291.000019,0.589403,32.011333,1333.376680,RICE YIELD (Kg per ha),1486.0,15,3
4,1,1997,chhattisgarh,48.123741,0.293813,0.545439,32.280,888.940025,28.8200,22.533164,35.769900,35.180497,35.144383,0.286679,0.287350,0.284562,0.506468,0.518796,0.525718,32.142,32.528,31.282,901.209999,1192.210018,1593.490021,29.3425,28.6325,28.6950,0.138,-12.269974,12.353841,31.984000,1228.970013,RICE YIELD (Kg per ha),1265.0,15,3


## CELL 8 — XGBoost Training with Walk-Forward CV
### Log Transform + No Shuffle = No Data Leakage

In [23]:
# ── XGBoost params — yearly pattern ke liye tuned ─────────────────────────
# min_child_weight aur reg_lambda overfitting rokenge
# subsample + colsample_bytree = variance reduce
XGB_PARAMS = {
    'n_estimators'     : 500,
    'max_depth'        : 6,
    'learning_rate'    : 0.05,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'min_child_weight' : 5,      # small datasets mein overfitting rokta hai
    'reg_lambda'       : 1.5,    # L2 regularization
    'reg_alpha'        : 0.5,    # L1 regularization
    'objective'        : 'reg:squarederror',
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbosity'        : 0
}

X_all  = df_melted[ALL_FEATURES]
y_raw  = df_melted['Yield']
y_log  = np.log1p(y_raw)    # Log Transform — skewed yields fix

# ── Walk-Forward CV loop ──────────────────────────────────────────────────
fold_results = []
all_actuals  = []
all_preds    = []
all_years_te = []

print('🚀 Walk-Forward Training Shuru...')
print(f'   {"Fold":<6} {"Test Year":<12} {"Train Rows":<12} {"Test Rows":<12} {"R2":>8} {"MAE":>10} {"MAPE":>8}')
print('   ' + '-' * 72)

for fold_num, (train_idx, test_idx) in enumerate(splits):
    X_train = X_all.loc[train_idx]
    X_test  = X_all.loc[test_idx]
    y_train = y_log.loc[train_idx]
    y_test_log = y_log.loc[test_idx]
    y_test_raw = y_raw.loc[test_idx]

    test_year = df_melted.loc[test_idx, 'year'].min()

    # Train
    model_fold = xgb.XGBRegressor(**XGB_PARAMS)
    model_fold.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test_log)],
        verbose=False
    )

    # Predict + inverse log
    preds_log = model_fold.predict(X_test)
    preds_raw = np.maximum(np.expm1(preds_log), 0)

    r2   = r2_score(y_test_raw, preds_raw)
    mae  = mean_absolute_error(y_test_raw, preds_raw)
    mape = np.mean(np.abs((y_test_raw - preds_raw) / y_test_raw.clip(lower=1))) * 100

    fold_results.append({'fold': fold_num+1, 'test_year': test_year,
                          'r2': r2, 'mae': mae, 'mape': mape,
                          'train_rows': len(train_idx), 'test_rows': len(test_idx)})
    all_actuals.extend(y_test_raw.tolist())
    all_preds.extend(preds_raw.tolist())
    all_years_te.extend([test_year] * len(test_idx))

    print(f'   {fold_num+1:<6} {test_year:<12} {len(train_idx):<12} {len(test_idx):<12} {r2:>8.4f} {mae:>10.1f} {mape:>7.1f}%')

# ── Aggregate metrics ─────────────────────────────────────────────────────
results_df = pd.DataFrame(fold_results)
overall_r2   = r2_score(all_actuals, all_preds)
overall_mae  = mean_absolute_error(all_actuals, all_preds)
overall_mape = np.mean(np.abs((np.array(all_actuals) - np.array(all_preds))
                              / np.array(all_actuals).clip(1))) * 100

print('   ' + '=' * 72)
print(f'   {"OVERALL":<6} {"":<12} {"":<12} {"":<12} {overall_r2:>8.4f} {overall_mae:>10.1f} {overall_mape:>7.1f}%')
print()
print(f'📊 Comparison with Old Random Split Model:')
print(f'   Old R2 (Random, No Lag): 0.8052  →  New R2 (WalkFwd + Lag): {overall_r2:.4f}')
print(f'   Old MAE               : 393.66  →  New MAE               : {overall_mae:.2f}')

🚀 Walk-Forward Training Shuru...
   Fold   Test Year    Train Rows   Test Rows          R2        MAE     MAPE
   ------------------------------------------------------------------------
   1      2003         45002        4530           0.8333      329.0    45.2%
   2      2004         49532        4552           0.8413      316.5    36.5%
   3      2005         54084        4611           0.8302      314.1    33.2%
   4      2006         58695        4578           0.8458      316.7    37.2%
   5      2007         63273        4483           0.7958      373.5    33.9%
   6      2008         67756        4389           0.8062      342.4    38.4%
   7      2009         72145        4222           0.8049      352.7    60.8%
   8      2010         76367        4373           0.7924      392.9    30.8%
   9      2011         80740        4283           0.7945      395.1    32.4%
   10     2012         85023        4321           0.7974      396.9    35.4%
   11     2013         89344     

In [25]:
import joblib
import xgboost as xgb

print("\n🚀 Final Model Training on Full Data...")

final_model = xgb.XGBRegressor(**XGB_PARAMS)

final_model.fit(X_all, y_log)

print("✅ Final model trained on full dataset")


🚀 Final Model Training on Full Data...
✅ Final model trained on full dataset


In [26]:
MODEL_PATH = "krishi_xgb_final.pkl"

joblib.dump(final_model, MODEL_PATH)

print(f"💾 Model saved at: {MODEL_PATH}")

💾 Model saved at: krishi_xgb_final.pkl


In [24]:
joblib.dump(final_model, 'krishi_twin_xgb_model.pkl')
print('✅ Model saved as krishi_twin_xgb_model.pkl')

NameError: name 'final_model' is not defined

### GridSearchCV

In [27]:
df_melted.to_csv('KrishiTwin_WalkForward_Engineered.csv', index=False)
print('✅ Walk-Forward Engineered data saved: KrishiTwin_WalkForward_Engineered.csv')

✅ Walk-Forward Engineered data saved: KrishiTwin_WalkForward_Engineered.csv


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error

# ── 1. Features & Data Setup ─────────────────────────────────────────────
# Base features
base_model_features = [
    'year', 'State_Encoded', 'Crop_Encoded', 'NPK_Intensity_KgHa',
    'Irrigation_Intensity_Ratio', 'WDI', 'Kharif_Avg_MaxTemp',
    'Kharif_Total_Rain', 'Rabi_Avg_MaxTemp', 'District_Soil_Health_Score',
]

# Lag + Delta + Rolling features
lag_model_features = [c for c in df_melted.columns
                      if ('_Lag' in c or '_Delta' in c or '_Roll' in c)
                      and c in df_melted.columns]

ALL_FEATURES = base_model_features + lag_model_features
ALL_FEATURES = [f for f in ALL_FEATURES if f in df_melted.columns]

print(f'✅ Total features for model: {len(ALL_FEATURES)}')

X_all  = df_melted[ALL_FEATURES]
y_raw  = df_melted['Yield']
y_log  = np.log1p(y_raw)    # Log Transform — skewed yields fix

# ── 2. Walk-Forward Splits (Assuming 'splits' variable is already generated)
# splits, all_years = walk_forward_split(df_melted, year_col='year', min_train_years=10)

# ── 3. HYPERPARAMETER TUNING (GridSearchCV with ZERO Leakage) ────────────
print('\n⏳ Grid Search chal raha hai (with Walk-Forward Validation)...')

# Hum yahan alag-alag variations try karenge
param_grid = {
    'n_estimators'     : [300, 500],
    'max_depth'        : [4, 6, 8],
    'learning_rate'    : [0.01, 0.05, 0.1],
    'subsample'        : [0.8],           # Fixed to save time
    'colsample_bytree' : [0.8],           # Fixed to save time
    'min_child_weight' : [3, 5, 7],
    'reg_lambda'       : [1.0, 1.5, 2.0]
}

# Base Model (n_jobs=1 rakha h taaki GridSearch apna multiprocessing handle kare)
xgb_base = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=1
)

# MAGIC STEP: cv=splits pass karne se 0 data leakage hoti hai!
gs = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=splits,           # <-- Ye step ensure karega ki time sequence break na ho!
    scoring='r2',
    n_jobs=-1,           # Saare CPU cores use karo fast processing ke liye
    verbose=1
)

# Fit Grid Search
gs.fit(X_all, y_log)

# Extract Best Parameters
BEST_PARAMS = gs.best_params_
print(f"\n✅ GridSearch Complete! Best Params Found:")
for k, v in BEST_PARAMS.items():
    print(f"   {k}: {v}")

# ── 4. WALK-FORWARD CV LOOP (Using Tuned Parameters) ──────────────────────
# Ab hum best parameters ke saath tumhari fold-by-fold summary generate karenge

# Add remaining static params to best_params
BEST_PARAMS['objective'] = 'reg:squarederror'
BEST_PARAMS['random_state'] = 42
BEST_PARAMS['n_jobs'] = -1

fold_results = []
all_actuals  = []
all_preds    = []
all_years_te = []

print('\n🚀 Walk-Forward Training Shuru (With Tuned Model)...')
print(f'   {"Fold":<6} {"Test Year":<12} {"Train Rows":<12} {"Test Rows":<12} {"R2":>8} {"MAE":>10} {"MAPE":>8}')
print('   ' + '-' * 72)

for fold_num, (train_idx, test_idx) in enumerate(splits):
    X_train = X_all.loc[train_idx]
    X_test  = X_all.loc[test_idx]
    y_train = y_log.loc[train_idx]
    y_test_log = y_log.loc[test_idx]
    y_test_raw = y_raw.loc[test_idx]

    test_year = df_melted.loc[test_idx, 'year'].min()

    # Train model using Best Parameters
    model_fold = xgb.XGBRegressor(**BEST_PARAMS)
    model_fold.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test_log)],
        verbose=False
    )

    # Predict + inverse log (raw Yield mein badalna)
    preds_log = model_fold.predict(X_test)
    preds_raw = np.maximum(np.expm1(preds_log), 0)

    # Calculate Metrics
    r2   = r2_score(y_test_raw, preds_raw)
    mae  = mean_absolute_error(y_test_raw, preds_raw)
    mape = np.mean(np.abs((y_test_raw - preds_raw) / y_test_raw.clip(lower=1))) * 100

    fold_results.append({
        'fold': fold_num+1, 'test_year': test_year,
        'r2': r2, 'mae': mae, 'mape': mape,
        'train_rows': len(train_idx), 'test_rows': len(test_idx)
    })

    all_actuals.extend(y_test_raw.tolist())
    all_preds.extend(preds_raw.tolist())
    all_years_te.extend([test_year] * len(test_idx))

    print(f'   {fold_num+1:<6} {test_year:<12} {len(train_idx):<12} {len(test_idx):<12} {r2:>8.4f} {mae:>10.1f} {mape:>7.1f}%')

# ── 5. Aggregate metrics ──────────────────────────────────────────────────
results_df = pd.DataFrame(fold_results)
overall_r2   = r2_score(all_actuals, all_preds)
overall_mae  = mean_absolute_error(all_actuals, all_preds)
overall_mape = np.mean(np.abs((np.array(all_actuals) - np.array(all_preds))
                              / np.array(all_actuals).clip(1))) * 100

print('   ' + '=' * 72)
print(f'   {"OVERALL":<6} {"":<12} {"":<12} {"":<12} {overall_r2:>8.4f} {overall_mae:>10.1f} {overall_mape:>7.1f}%')
print()

✅ Total features for model: 33

⏳ Grid Search chal raha hai (with Walk-Forward Validation)...
Fitting 13 folds for each of 162 candidates, totalling 2106 fits


KeyboardInterrupt: 

### GPU;-

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error

# ── 1. Features & Data Setup ─────────────────────────────────────────────
base_model_features = [
    'year', 'State_Encoded', 'Crop_Encoded', 'NPK_Intensity_KgHa',
    'Irrigation_Intensity_Ratio', 'WDI', 'Kharif_Avg_MaxTemp',
    'Kharif_Total_Rain', 'Rabi_Avg_MaxTemp', 'District_Soil_Health_Score',
]

lag_model_features = [c for c in df_melted.columns
                      if ('_Lag' in c or '_Delta' in c or '_Roll' in c)
                      and c in df_melted.columns]

ALL_FEATURES = base_model_features + lag_model_features
ALL_FEATURES = [f for f in ALL_FEATURES if f in df_melted.columns]

print(f'✅ Total features for model: {len(ALL_FEATURES)}')

X_all  = df_melted[ALL_FEATURES]
y_raw  = df_melted['Yield']
y_log  = np.log1p(y_raw)

# ── 2. Walk-Forward Splits (Assuming 'splits' variable is already generated)
# splits, all_years = walk_forward_split(df_melted, year_col='year', min_train_years=10)

# ── 3. HYPERPARAMETER TUNING (GPU Accelerated GridSearchCV) ────────────
print('\n⏳ GPU par Grid Search chal raha hai (with Walk-Forward Validation)...')

param_grid = {
    'n_estimators'     : [300, 500],
    'max_depth'        : [4, 6, 8],
    'learning_rate'    : [0.01, 0.05, 0.1],
    'subsample'        : [0.8],
    'colsample_bytree' : [0.8],
    'min_child_weight' : [3, 5, 7],
    'reg_lambda'       : [1.0, 1.5, 2.0]
}

# 🚀 GPU ACTIVATION STEP 🚀
xgb_base = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    tree_method='hist', # Modern XGBoost GPU algorithm
    device='cuda',      # Order to use GPU
    # NOTE: Agar error aaye ki 'device' unexpected keyword hai, toh in dono lines ko hata kar
    # sirf `tree_method='gpu_hist'` likh dena (purane versions ke liye).
)

gs = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=splits,
    scoring='r2',
    n_jobs=1,           # ⚠️ GPU CRASH SE BACHNE KE LIYE ISE 1 HI RAKHNA
    verbose=1
)

# Fit Grid Search (Ye ab tumhare GPU ko heat up karega!)
gs.fit(X_all, y_log)

BEST_PARAMS = gs.best_params_
print(f"\n✅ GPU GridSearch Complete! Best Params Found:")
for k, v in BEST_PARAMS.items():
    print(f"   {k}: {v}")

# ── 4. WALK-FORWARD CV LOOP (Using Tuned Parameters on GPU) ──────────────

# Add static GPU params to best_params
BEST_PARAMS['objective'] = 'reg:squarederror'
BEST_PARAMS['random_state'] = 42
BEST_PARAMS['tree_method'] = 'hist'
BEST_PARAMS['device'] = 'cuda'

fold_results = []
all_actuals  = []
all_preds    = []
all_years_te = []

print('\n🚀 Walk-Forward Training Shuru (With Tuned Model on GPU)...')
print(f'   {"Fold":<6} {"Test Year":<12} {"Train Rows":<12} {"Test Rows":<12} {"R2":>8} {"MAE":>10} {"MAPE":>8}')
print('   ' + '-' * 72)

for fold_num, (train_idx, test_idx) in enumerate(splits):
    X_train = X_all.loc[train_idx]
    X_test  = X_all.loc[test_idx]
    y_train = y_log.loc[train_idx]
    y_test_log = y_log.loc[test_idx]
    y_test_raw = y_raw.loc[test_idx]

    test_year = df_melted.loc[test_idx, 'year'].min()

    # Train model using Best Parameters
    model_fold = xgb.XGBRegressor(**BEST_PARAMS)
    model_fold.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test_log)],
        verbose=False
    )

    # Predict + inverse log
    preds_log = model_fold.predict(X_test)
    preds_raw = np.maximum(np.expm1(preds_log), 0)

    # Calculate Metrics
    r2   = r2_score(y_test_raw, preds_raw)
    mae  = mean_absolute_error(y_test_raw, preds_raw)
    mape = np.mean(np.abs((y_test_raw - preds_raw) / y_test_raw.clip(lower=1))) * 100

    fold_results.append({
        'fold': fold_num+1, 'test_year': test_year,
        'r2': r2, 'mae': mae, 'mape': mape,
        'train_rows': len(train_idx), 'test_rows': len(test_idx)
    })

    all_actuals.extend(y_test_raw.tolist())
    all_preds.extend(preds_raw.tolist())
    all_years_te.extend([test_year] * len(test_idx))

    print(f'   {fold_num+1:<6} {test_year:<12} {len(train_idx):<12} {len(test_idx):<12} {r2:>8.4f} {mae:>10.1f} {mape:>7.1f}%')

# ── 5. Aggregate metrics ──────────────────────────────────────────────────
overall_r2   = r2_score(all_actuals, all_preds)
overall_mae  = mean_absolute_error(all_actuals, all_preds)
overall_mape = np.mean(np.abs((np.array(all_actuals) - np.array(all_preds))
                              / np.array(all_actuals).clip(1))) * 100

print('   ' + '=' * 72)
print(f'   {"OVERALL":<6} {"":<12} {"":<12} {"":<12} {overall_r2:>8.4f} {overall_mae:>10.1f} {overall_mape:>7.1f}%')
print()

### Randomized CV with GPUs:-

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error

# ── 1. Features & Data Setup ─────────────────────────────────────────────
# Base features
base_model_features = [
    'year', 'State_Encoded', 'Crop_Encoded', 'NPK_Intensity_KgHa',
    'Irrigation_Intensity_Ratio', 'WDI', 'Kharif_Avg_MaxTemp',
    'Kharif_Total_Rain', 'Rabi_Avg_MaxTemp', 'District_Soil_Health_Score',
]

# Lag + Delta + Rolling features
lag_model_features = [c for c in df_melted.columns
                      if ('_Lag' in c or '_Delta' in c or '_Roll' in c)
                      and c in df_melted.columns]

ALL_FEATURES = base_model_features + lag_model_features
ALL_FEATURES = [f for f in ALL_FEATURES if f in df_melted.columns]

print(f'✅ Total features for model: {len(ALL_FEATURES)}')

X_all  = df_melted[ALL_FEATURES]
y_raw  = df_melted['Yield']
y_log  = np.log1p(y_raw)    # Log Transform — skewed yields fix

# ── 2. Walk-Forward Splits (Assuming 'splits' variable is already generated)
# splits, all_years = walk_forward_split(df_melted, year_col='year', min_train_years=10)

# ── 3. HYPERPARAMETER TUNING (RandomizedSearchCV + GPU ENABLED) ────────────
print('\n⏳ Randomized Search chal raha hai (GPU + Walk-Forward Validation)...')

param_grid = {
    'n_estimators'     : [300, 500, 700],
    'max_depth'        : [4, 6, 8, 10],
    'learning_rate'    : [0.01, 0.05, 0.1, 0.2],
    'subsample'        : [0.7, 0.8, 0.9],
    'colsample_bytree' : [0.7, 0.8, 0.9],
    'min_child_weight' : [3, 5, 7],
    'reg_lambda'       : [1.0, 1.5, 2.0, 5.0]
}

# 🚀 MAGIC STEP: Enabling GPU in Base Model
xgb_base = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    tree_method='hist',  # Naye XGBoost versions ke liye GPU use karne ka tareeqa
    device='cuda'        # GPU enable karne ki command
    # NOTE: Agar tumhara XGBoost version purana (v1.x) hai, toh in dono lines ko hatakar sirf yeh likhna:
    # tree_method='gpu_hist'
)

rs = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=25,
    cv=splits,
    scoring='r2',
    n_jobs=1,            # ⚠️ CRITICAL: GPU use karte waqt isko 1 hi rakhna varna GPU Out-of-Memory (OOM) ho jayega!
    verbose=1,
    random_state=42
)

# Fit Randomized Search
rs.fit(X_all, y_log)

# Extract Best Parameters
BEST_PARAMS = rs.best_params_
print(f"\n✅ RandomizedSearch Complete! Best Params Found:")
for k, v in BEST_PARAMS.items():
    print(f"   {k}: {v}")

# ── 4. WALK-FORWARD CV LOOP (Using Tuned Parameters + GPU) ────────────────
# Add remaining static params to best_params so fold-training also uses GPU
BEST_PARAMS['objective'] = 'reg:squarederror'
BEST_PARAMS['random_state'] = 42
BEST_PARAMS['tree_method'] = 'hist'  # Keep GPU active for final walk-forward
BEST_PARAMS['device'] = 'cuda'       # Keep GPU active for final walk-forward
# BEST_PARAMS['tree_method'] = 'gpu_hist' # Use this if on older XGBoost

fold_results = []
all_actuals  = []
all_preds    = []
all_years_te = []

print('\n🚀 Walk-Forward Training Shuru (With Tuned Model on GPU)...')
print(f'   {"Fold":<6} {"Test Year":<12} {"Train Rows":<12} {"Test Rows":<12} {"R2":>8} {"MAE":>10} {"MAPE":>8}')
print('   ' + '-' * 72)

for fold_num, (train_idx, test_idx) in enumerate(splits):
    X_train = X_all.loc[train_idx]
    X_test  = X_all.loc[test_idx]
    y_train = y_log.loc[train_idx]
    y_test_log = y_log.loc[test_idx]
    y_test_raw = y_raw.loc[test_idx]

    test_year = df_melted.loc[test_idx, 'year'].min()

    # Train model using Best Parameters (which now include GPU settings)
    model_fold = xgb.XGBRegressor(**BEST_PARAMS)
    model_fold.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test_log)],
        verbose=False
    )

    # Predict + inverse log (raw Yield mein badalna)
    preds_log = model_fold.predict(X_test)
    preds_raw = np.maximum(np.expm1(preds_log), 0)

    # Calculate Metrics
    r2   = r2_score(y_test_raw, preds_raw)
    mae  = mean_absolute_error(y_test_raw, preds_raw)
    mape = np.mean(np.abs((y_test_raw - preds_raw) / y_test_raw.clip(lower=1))) * 100

    fold_results.append({
        'fold': fold_num+1, 'test_year': test_year,
        'r2': r2, 'mae': mae, 'mape': mape,
        'train_rows': len(train_idx), 'test_rows': len(test_idx)
    })

    all_actuals.extend(y_test_raw.tolist())
    all_preds.extend(preds_raw.tolist())
    all_years_te.extend([test_year] * len(test_idx))

    print(f'   {fold_num+1:<6} {test_year:<12} {len(train_idx):<12} {len(test_idx):<12} {r2:>8.4f} {mae:>10.1f} {mape:>7.1f}%')

# ── 5. Aggregate metrics ──────────────────────────────────────────────────
results_df = pd.DataFrame(fold_results)
overall_r2   = r2_score(all_actuals, all_preds)
overall_mae  = mean_absolute_error(all_actuals, all_preds)
overall_mape = np.mean(np.abs((np.array(all_actuals) - np.array(all_preds))
                              / np.array(all_actuals).clip(1))) * 100

print('   ' + '=' * 72)
print(f'   {"OVERALL":<6} {"":<12} {"":<12} {"":<12} {overall_r2:>8.4f} {overall_mae:>10.1f} {overall_mape:>7.1f}%')
print()

✅ Total features for model: 33

⏳ Randomized Search chal raha hai (GPU + Walk-Forward Validation)...
Fitting 13 folds for each of 25 candidates, totalling 325 fits


## CELL 9 — Final Model (Poore Data par Train)

In [ ]:
# Walk-Forward CV ke baad metrics mil gayi
# Ab final production model poore data par train karo
# (Sabse zyada data = sabse achha model)

print('🔨 Final Production Model Training on Full Data...')

final_model = xgb.XGBRegressor(**XGB_PARAMS)
final_model.fit(X_all, y_log, verbose=False)

# Save
joblib.dump(final_model, 'krishi_twin_xgb_model_complete.pkl')
print('✅ Final model saved: krishi_twin_xgb_model.pkl')

# Quick sanity check — train data par hi predict karke dekho
sanity_preds = np.maximum(np.expm1(final_model.predict(X_all)), 0)
sanity_r2    = r2_score(y_raw, sanity_preds)
print(f'   Train R2 (sanity): {sanity_r2:.4f}  (should be high — this is train data)')
print(f'   Walk-Fwd R2      : {overall_r2:.4f}  ← Yeh actual generalization metric hai')

## CELL 10 — Feature Importance

In [ ]:
importances = final_model.feature_importances_
feat_imp_df = pd.DataFrame({'Feature': ALL_FEATURES, 'Importance': importances})\
                .sort_values('Importance', ascending=False).reset_index(drop=True)

print('📊 Top 20 Feature Importances:')
print(feat_imp_df.head(20).to_string(index=False))

# Lag features ka combined importance
lag_imp   = feat_imp_df[feat_imp_df['Feature'].str.contains('_Lag|_Delta|_Roll')]['Importance'].sum()
base_imp  = feat_imp_df[~feat_imp_df['Feature'].str.contains('_Lag|_Delta|_Roll')]['Importance'].sum()
print(f'\n🔍 Lag/Delta/Roll features total importance : {lag_imp:.4f} ({lag_imp*100:.1f}%)')
print(f'   Base features total importance          : {base_imp:.4f} ({base_imp*100:.1f}%)')
print()
if lag_imp > 0.15:
    print('✅ Lag features significant hain — model yearly pattern seekh raha hai!')
else:
    print('⚠️  Lag importance kam hai — data mein zyada districts add karo for better learning')

## CELL 11 — Plots

In [ ]:
fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Plot 1: R2 per fold (yearly trend) ───────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(results_df['test_year'], results_df['r2'],
         marker='o', color='#2ecc71', linewidth=2, markersize=6)
ax1.axhline(y=overall_r2, color='navy', linestyle='--', alpha=0.7, label=f'Overall R2={overall_r2:.4f}')
ax1.axhline(y=0.8052,     color='red',  linestyle='--', alpha=0.5, label='Old Random R2=0.8052')
ax1.set_title('R² Score per Fold (Walk-Forward)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Test Year')
ax1.set_ylabel('R² Score')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# ── Plot 2: MAE per fold ──────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(results_df['test_year'], results_df['mae'],
        color='#3498db', alpha=0.8, edgecolor='white')
ax2.axhline(y=393.66, color='red', linestyle='--', alpha=0.7, label='Old MAE=393.66')
ax2.axhline(y=overall_mae, color='green', linestyle='--', alpha=0.7, label=f'New MAE={overall_mae:.1f}')
ax2.set_title('MAE per Fold', fontsize=12, fontweight='bold')
ax2.set_xlabel('Test Year')
ax2.set_ylabel('MAE (Kg/ha)')
ax2.legend(fontsize=8)
ax2.tick_params(axis='x', rotation=45)

# ── Plot 3: Actual vs Predicted (all folds combined) ─────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(all_actuals, all_preds, alpha=0.3, color='teal', s=10)
lim = max(max(all_actuals), max(all_preds))
ax3.plot([0, lim], [0, lim], '--r', linewidth=1.5)
ax3.set_title(f'Actual vs Predicted\n(Walk-Forward, All Folds)\nR²={overall_r2:.4f}  MAE={overall_mae:.1f}',
              fontsize=11, fontweight='bold')
ax3.set_xlabel('Actual Yield (Kg/ha)')
ax3.set_ylabel('Predicted Yield (Kg/ha)')

# ── Plot 4: Feature Importance (Top 15) ──────────────────────────────────
ax4 = fig.add_subplot(gs[1, :])
top15 = feat_imp_df.head(15)
colors_fi = ['#e74c3c' if ('_Lag' in f or '_Delta' in f or '_Roll' in f)
             else '#2980b9' for f in top15['Feature']]
bars = ax4.barh(top15['Feature'][::-1], top15['Importance'][::-1],
                color=colors_fi[::-1], edgecolor='white')
ax4.set_title('Top 15 Feature Importances\n(Red = Lag/Temporal features | Blue = Base features)',
              fontsize=12, fontweight='bold')
ax4.set_xlabel('Importance')
for bar, val in zip(bars, top15['Importance'][::-1]):
    ax4.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=8)

fig.suptitle('KrishiTwin — Walk-Forward CV Results\n(Data Leakage Fixed + Lag Features for Yearly Pattern)',
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('KrishiTwin_WalkForward_Results.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
print('✅ Plot saved: KrishiTwin_WalkForward_Results.png')

## CELL 12 — Summary

In [ ]:
print('=' * 65)
print('  KRISHITWIN — FINAL RESULTS SUMMARY')
print('=' * 65)
print(f'  {"":<35} {"R2":>8} {"MAE":>10} {"MAPE":>8}')
print('  ' + '-' * 63)
print(f'  {"Old Model (Random Split, No Lag)":<35} {0.8052:>8.4f} {393.66:>10.2f} {"~30%":>8}')
print(f'  {"New Model (Walk-Forward + Lag)":<35} {overall_r2:>8.4f} {overall_mae:>10.2f} {overall_mape:>7.1f}%')
print('  ' + '=' * 63)
print()
print('  Problems Fixed:')
print('  ✅ Data Leakage      — Walk-Forward split, koi shuffle nahi')
print('  ✅ Yearly Pattern     — Lag1/Lag2/Lag3 + Delta + Rolling features')
print('  ✅ Skewed Yields      — Log Transform')
print('  ✅ Overfitting        — min_child_weight + L1/L2 regularization')
print()
print('  Saved Files:')
print('  📦 krishi_twin_xgb_model.pkl')
print('  📦 crop_encoder.pkl')
print('  📦 state_encoder.pkl')
print('  📊 KrishiTwin_WalkForward_Results.png')
print('=' * 65)